<a href="https://colab.research.google.com/github/MoeTahech/real-fake-face-detection-thesis/blob/main/notebooks/07b_diffusionface_external_check_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07b — Phase 7 supplementary check: DiffusionFace (external diffusion test)

**This notebook does NOT replace Phase 7.** Notebook 07 and its results stay exactly as they are.
This is a separate, evaluation-only check, run because the Phase 7 diffusion set has no real images of its
own and a file-format shortcut (Phase 13: PNG 512x512 fakes vs JPEG 256x256 reals).

**Dataset:** DiffusionFace (Chen et al., 2024, arXiv:2403.18471), downloaded from Zenodo
(part 1: DOI 10.5281/zenodo.10865300, part 2: DOI 10.5281/zenodo.10865065; CC BY 4.0, research use).
It has its own real faces (Multi-Modal-CelebA-HQ) and fakes from several diffusion models, all 256x256 according to the paper.

**No training.** The saved Phase 2 and Phase 6 checkpoints are only evaluated, so neither has seen any DiffusionFace image.

The notebook has three parts. Run it in order and **stop after each part** to check the output:
- **Part A:** download, extract and inspect the files (what is inside? which format?).
- **Part B:** format and pixel-statistics audit (same idea as Phase 13): can file metadata alone separate real from fake?
- **Part C:** evaluate the models on two pairings (DiffusionFace reals / your FFHQ reals), with and without JPEG-75 normalization.

Requires the checkpoints `best_model_finetuned.pth` (Phase 2) and `best_model_mixed.pth` (Phase 6) in Drive.
Runtime -> Change runtime type -> GPU. Needs about 15 GB of free disk in the Colab session.

### Step 1 — Confirm you have a GPU

In [ ]:
!nvidia-smi

Mon Oct 19 11:42:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 2 — Mount Google Drive and list the models to evaluate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/thesis_checkpoints'

MODELS = {
    "Phase 2 (single-source)": f"{CHECKPOINT_DIR}/best_model_finetuned.pth",
    "Phase 6 (mixed-source)":  f"{CHECKPOINT_DIR}/best_model_mixed.pth",

}
for name, path in MODELS.items():
    print(f"{name:<28} found: {os.path.exists(path)}")

Mounted at /content/drive
Phase 2 (single-source)      found: True
Phase 6 (mixed-source)       found: True


## PART A — Download and inspect

### Step 3 — Download the files (about 13 GB) and verify their checksums

Only 4 of the many DiffusionFace archives are downloaded:
- `Real` and `MM_CelebA`: two candidate real-image sets (we inspect both, then choose one in Step 5)
- `SD15_T2I`: Stable Diffusion v1.5 text-to-image fakes (one of the three variants)
- `SD21_T2I`: Stable Diffusion v2.1 text-to-image fakes (one of the three variants)

The checksums come from the Zenodo record pages. Downloads can be slow; `wget -c` resumes if interrupted
(just re-run the cell).

In [ ]:
import os, hashlib

WORK = "/content/diffface"
os.makedirs(WORK, exist_ok=True)

P1 = "https://zenodo.org/records/10865300/files/{}?download=1"   # part 1
P2 = "https://zenodo.org/records/10865065/files/{}?download=1"   # part 2

# key: (url, file name, md5 listed on Zenodo)
FILES = {
    "Real":      (P1.format("Real.tar"),      "Real.tar",      "72379bcecf6382d7999cce59b33dfcdc"),
    "MM_CelebA": (P1.format("MM_CelebA.tar"), "MM_CelebA.tar", "1255f51499c52ed2b47841e2d45ac869"),
    "SD15_T2I":  (P2.format("stable_diffusion_v_1_5_text2img_p3g7.tar"),
                  "stable_diffusion_v_1_5_text2img_p3g7.tar", "37a92f1973d6cc8941c8d98a195c09b4"),
    "SD21_T2I":  (P2.format("stable_diffusion_v_2_1_text2img_p0g5.tar"),
                  "stable_diffusion_v_2_1_text2img_p0g5.tar", "55bd322166cfdb3bda9aa05cb5a545aa"),
}

def md5_of(path, chunk=1 << 24):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

for key, (url, fname, md5) in FILES.items():
    path = f"{WORK}/{fname}"
    if not os.path.exists(path):
        print("Downloading", fname)
        get_ipython().system(f'wget -c -q --show-progress -O "{path}" "{url}"')
    ok = md5_of(path) == md5
    print(f"{fname:<48} md5 {'OK' if ok else 'MISMATCH - delete the file and re-run this cell'}")

/content/diffface/R 100%[===================>]   3.23G  3.03MB/s    in 15m 10s 
Real.tar                                         md5 OK
/content/diffface/M 100%[===================>]   2.99G  9.78MB/s    in 16m 59s 
MM_CelebA.tar                                    md5 OK
/content/diffface/s 100%[===================>]   2.78G  2.76MB/s    in 17m 51s 
stable_diffusion_v_1_5_text2img_p3g7.tar         md5 OK
/content/diffface/s 100%[===================>]   2.90G  2.79MB/s    in 16m 57s 
stable_diffusion_v_2_1_text2img_p0g5.tar         md5 OK


### Step 4 — Extract each archive into its own folder

In [ ]:
for key, (url, fname, md5) in FILES.items():
    dest = f"{WORK}/{key}"
    if not os.path.isdir(dest) or not os.listdir(dest):
        os.makedirs(dest, exist_ok=True)
        print("Extracting", fname)
        get_ipython().system(f'tar -xf "{WORK}/{fname}" -C "{dest}"')
print("Done.")
get_ipython().system(f'df -h /content | tail -1')

Extracting Real.tar
Extracting MM_CelebA.tar
Extracting stable_diffusion_v_1_5_text2img_p3g7.tar
Extracting stable_diffusion_v_2_1_text2img_p0g5.tar
Done.
overlay         113G   67G   47G  60% /


### Step 5 — Inspect what is inside (STOP and read this output)

For each folder: how many images, which file types, which sizes, and the top-level subfolders.
Look for: (1) whether `Real` and `MM_CelebA` are the same kind of images, (2) whether the fake folders
contain only fakes (no masks / reals mixed in), (3) whether real and fake use the **same file extension and size**.

In [ ]:
import random
from collections import Counter
from PIL import Image

IMG_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")

def find_all_images(root):
    found = []
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if f.lower().endswith(IMG_EXTS):
                found.append(os.path.join(dirpath, f))
    return sorted(found)

def describe(root, n_sample=200, seed=0):
    paths = find_all_images(root)
    print(f"\n=== {os.path.basename(root)}: {len(paths)} images ===")
    if not paths:
        return
    exts = Counter(os.path.splitext(p)[1].lower() for p in paths)
    top = Counter(os.path.relpath(os.path.dirname(p), root).split(os.sep)[0] for p in paths)
    rng = random.Random(seed)
    sample = rng.sample(paths, min(n_sample, len(paths)))
    sizes, modes, fmts = Counter(), Counter(), Counter()
    for p in sample:
        with Image.open(p) as im:
            sizes[im.size] += 1; modes[im.mode] += 1; fmts[im.format] += 1
    print("extensions      :", dict(exts))
    print("sampled formats :", dict(fmts))
    print("sampled sizes   :", dict(sizes))
    print("sampled modes   :", dict(modes))
    print("top-level dirs  :", dict(list(top.items())[:10]))
    print("example paths   :", paths[:2])

for key in FILES:
    describe(f"{WORK}/{key}")


=== Real: 30000 images ===
extensions      : {'.png': 30000}
sampled formats : {'PNG': 200}
sampled sizes   : {(256, 256): 200}
sampled modes   : {'RGB': 200}
top-level dirs  : {'Real': 30000}
example paths   : ['/content/diffface/Real/Real/0.png', '/content/diffface/Real/Real/1.png']

=== MM_CelebA: 0 images ===

=== SD15_T2I: 30000 images ===
extensions      : {'.png': 30000}
sampled formats : {'PNG': 200}
sampled sizes   : {(256, 256): 200}
sampled modes   : {'RGB': 200}
top-level dirs  : {'stable_diffusion_v_1_5_text2img_p3g7': 30000}
example paths   : ['/content/diffface/SD15_T2I/stable_diffusion_v_1_5_text2img_p3g7/aligned/0.png', '/content/diffface/SD15_T2I/stable_diffusion_v_1_5_text2img_p3g7/aligned/1.png']

=== SD21_T2I: 30000 images ===
extensions      : {'.png': 30000}
sampled formats : {'PNG': 200}
sampled sizes   : {(256, 256): 200}
sampled modes   : {'RGB': 200}
top-level dirs  : {'stable_diffusion_v_2_1_text2img_p0g5': 30000}
example paths   : ['/content/diffface/SD21_

**Checkpoint A.** Before continuing, check the output above and decide which folder is the real-image set
(`Real` or `MM_CelebA`). Prefer the one whose images have the **same size and processing as the fakes** (256x256).
If you are unsure, send me the Step 5 output and stop here.

## PART B — Format / shortcut audit

### Step 6 — Choose the sets and sample images
Set `REAL_KEY` after reading the Step 5 output. The sample is random but fixed by `SEED`.

In [ ]:
REAL_KEY  = "Real"                       # change to "MM_CelebA" if Step 5 shows it is the better real set
FAKE_KEYS = ["SD15_T2I", "SD21_T2I"]
N_PER_CLASS = 3000                       # images per class in each evaluation (raise to 5000 if time allows)
SEED = 7

rng = random.Random(SEED)

def sample_paths(key, n):
    paths = find_all_images(f"{WORK}/{key}")
    rng_local = random.Random(SEED)
    rng_local.shuffle(paths)
    return paths[:n]

real_paths = sample_paths(REAL_KEY, N_PER_CLASS)
fake_paths = {k: sample_paths(k, N_PER_CLASS) for k in FAKE_KEYS}
print("real  :", REAL_KEY, len(real_paths))
for k, v in fake_paths.items():
    print("fake  :", k, len(v))
n_eval = min([len(real_paths)] + [len(v) for v in fake_paths.values()])
print("Images per class used in every evaluation:", n_eval)

real  : Real 3000
fake  : SD15_T2I 3000
fake  : SD21_T2I 3000
Images per class used in every evaluation: 3000


### Step 7 — Can file facts or pixel statistics alone separate real from fake?
Same idea as Phase 13. A small classifier sees only file facts (size in KB, width, height, bytes per pixel,
PNG/JPEG flag, JPEG quantization) or only pixel statistics (brightness, contrast, sharpness), scored by 5-fold
cross-validated AUC. **0.5 = no information (good). Close to 1.0 = a shortcut exists.**

In [ ]:
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

def file_facts(p):
    st = os.path.getsize(p)
    with Image.open(p) as im:
        w, h = im.size
        fmt = im.format
        q = 0.0
        quant = getattr(im, "quantization", None)
        if fmt == "JPEG" and quant:
            q = float(np.mean([np.mean(t) for t in quant.values()]))
        gray = np.asarray(im.convert("L"), dtype=np.float32)
    lap = (gray[1:-1, 1:-1] * 4 - gray[:-2, 1:-1] - gray[2:, 1:-1]
           - gray[1:-1, :-2] - gray[1:-1, 2:])
    meta = [st / 1024, w, h, st / (w * h), float(fmt == "PNG"), float(fmt == "JPEG"), q]
    pix = [float(gray.mean()), float(gray.std()), float(lap.var())]
    return meta, pix, fmt

def collect(paths, n_audit):
    metas, pixs, fmts, kbs, dims = [], [], Counter(), [], Counter()
    for p in paths[:n_audit]:
        m, px, fmt = file_facts(p)
        metas.append(m); pixs.append(px); fmts[fmt] += 1; kbs.append(m[0]); dims[(int(m[1]), int(m[2]))] += 1
    return np.array(metas), np.array(pixs), fmts, float(np.median(kbs)), dims

def cv_auc(X, y):
    cv = StratifiedKFold(5, shuffle=True, random_state=0)
    s = cross_val_score(GradientBoostingClassifier(random_state=0), X, y, cv=cv, scoring="roc_auc")
    return s.mean(), s.std()

N_AUDIT = 1000
real_meta, real_pix, real_fmts, real_kb, real_dims = collect(real_paths, N_AUDIT)
print(f"REAL  {REAL_KEY}: formats {dict(real_fmts)}, median {real_kb:.0f} KB, sizes {dict(real_dims)}")

audit_rows = []
for k in FAKE_KEYS:
    f_meta, f_pix, f_fmts, f_kb, f_dims = collect(fake_paths[k], N_AUDIT)
    print(f"FAKE  {k}: formats {dict(f_fmts)}, median {f_kb:.0f} KB, sizes {dict(f_dims)}")
    n = min(len(real_meta), len(f_meta))
    y = np.array([1] * n + [0] * n)
    a_meta = cv_auc(np.vstack([real_meta[:n], f_meta[:n]]), y)
    a_pix = cv_auc(np.vstack([real_pix[:n], f_pix[:n]]), y)
    audit_rows.append((k, a_meta, a_pix))
    print(f"      file facts only AUC {a_meta[0]:.3f} +/- {a_meta[1]:.3f} | pixel stats only AUC {a_pix[0]:.3f} +/- {a_pix[1]:.3f}")
    if a_meta[0] > 0.9:
        print("      WARNING: file facts alone separate real from fake -> a format shortcut exists for this pairing.")

REAL  Real: formats {'PNG': 1000}, median 112 KB, sizes {(256, 256): 1000}
FAKE  SD15_T2I: formats {'PNG': 1000}, median 95 KB, sizes {(256, 256): 1000}
      file facts only AUC 0.796 +/- 0.018 | pixel stats only AUC 0.831 +/- 0.025
FAKE  SD21_T2I: formats {'PNG': 1000}, median 99 KB, sizes {(256, 256): 1000}
      file facts only AUC 0.731 +/- 0.021 | pixel stats only AUC 0.745 +/- 0.017


**Checkpoint B.** Read the audit. Rough guide (same reading as Phase 13):
- file-facts AUC close to **0.5** (up to about 0.6): clean, evaluation results are meaningful;
- file-facts AUC **above 0.9**: a format shortcut exists, so treat Part C results with the same caution as the old Phase 7
  and tell me (we would pick another subset or fall back to another dataset).

Send me the Step 5 and Step 7 outputs before running Part C if you want a second pair of eyes.

## PART C — Evaluate the saved models

### Step 8 (optional) — FFHQ real images for the second pairing
Pairing B scores DiffusionFace fakes against your original FFHQ real test images (the ones used in Phase 7).
Comparing pairing A (DiffusionFace reals) with pairing B (FFHQ reals) shows how much of any result comes from
unfamiliar real photos rather than unfamiliar fakes. Set `USE_FFHQ_REALS = False` to skip it (no Kaggle token needed then).

In [ ]:
USE_FFHQ_REALS = True

if USE_FFHQ_REALS:
    import os
    os.environ["KAGGLE_API_TOKEN"] = "YOUR_KAGGLE_API_TOKEN_HERE"   # replace locally; never commit a real token
    get_ipython().system("kaggle datasets download -d xhlulu/140k-real-and-fake-faces")
    get_ipython().system("unzip -q 140k-real-and-fake-faces.zip -d data_raw")

    def find_real_fake_folders(root):
        out = []
        for dirpath, dirnames, _ in os.walk(root):
            lower = [d.lower() for d in dirnames]
            if "real" in lower and "fake" in lower:
                out.append(dirpath)
        return out

    test_root = [x for x in find_real_fake_folders("data_raw") if "test" in x.lower()][0]
    ffhq_dir = [os.path.join(test_root, d) for d in os.listdir(test_root) if d.lower() == "real"][0]
    ffhq_all = sorted(os.path.join(ffhq_dir, f) for f in os.listdir(ffhq_dir))
    random.Random(SEED).shuffle(ffhq_all)
    ffhq_real_paths = ffhq_all[:N_PER_CLASS]
    print("FFHQ real test images available:", len(ffhq_all), "| used:", len(ffhq_real_paths))
else:
    ffhq_real_paths = []

Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
100% 3.75G/3.75G [00:48<00:00, 82.2MB/s]

FFHQ real test images available: 10000 | used: 3000


### Step 9 — Evaluation helpers

In [ ]:
import io
import numpy as np
from sklearn.metrics import roc_auc_score

def metrics_from_scores(p_real, y_real, n_boot=500, seed=0):
    """p_real: probability of 'real'; y_real: 1 = real image, 0 = fake image."""
    p_real = np.asarray(p_real); y_real = np.asarray(y_real)
    pred_real = (p_real >= 0.5).astype(int)
    acc = float((pred_real == y_real).mean())
    auc = float(roc_auc_score(y_real, p_real))
    fake_recall = float((pred_real[y_real == 0] == 0).mean())   # fakes correctly flagged as fake
    real_recall = float((pred_real[y_real == 1] == 1).mean())   # reals correctly accepted
    bal_acc = (fake_recall + real_recall) / 2
    # bootstrap CI for AUC (real and fake resampled separately)
    rng = np.random.default_rng(seed)
    pr, pf = p_real[y_real == 1], p_real[y_real == 0]
    aucs = []
    for _ in range(n_boot):
        r = rng.choice(pr, size=len(pr), replace=True)
        f = rng.choice(pf, size=len(pf), replace=True)
        aucs.append(roc_auc_score(np.r_[np.ones(len(r)), np.zeros(len(f))], np.r_[r, f]))
    lo, hi = np.percentile(aucs, [2.5, 97.5])
    return dict(acc=acc, auc=auc, auc_lo=float(lo), auc_hi=float(hi),
                fake_recall=fake_recall, real_recall=real_recall, bal_acc=bal_acc)

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
to_tensor = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), normalize])

class PathDataset(Dataset):
    """items: list of (path, is_real). jpeg_q: re-encode as JPEG at this quality before the usual preprocessing."""
    def __init__(self, items, jpeg_q=None):
        self.items, self.jpeg_q = items, jpeg_q
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, is_real = self.items[i]
        im = Image.open(path).convert("RGB")
        if self.jpeg_q:
            buf = io.BytesIO()
            im.save(buf, "JPEG", quality=self.jpeg_q)
            buf.seek(0)
            im = Image.open(buf).convert("RGB")
        return to_tensor(im), int(is_real)

def load_resnet(path):
    ckpt = torch.load(path, map_location=device)
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 2)
    model.load_state_dict(ckpt["model_state"])
    return model.to(device).eval(), ckpt["class_to_idx"]

@torch.no_grad()
def score_items(model, real_idx, items, jpeg_q=None):
    loader = DataLoader(PathDataset(items, jpeg_q), batch_size=64, shuffle=False, num_workers=2)
    probs, labels = [], []
    for x, y in loader:
        p = torch.softmax(model(x.to(device)), dim=1)[:, real_idx].cpu().numpy()
        probs.append(p); labels.append(y.numpy())
    return np.concatenate(probs), np.concatenate(labels)

def make_items(real_list, fake_list, n):
    return [(p, 1) for p in real_list[:n]] + [(p, 0) for p in fake_list[:n]]

### Step 10 — Run the evaluations
Two models x two pairings x two fake sets x two image variants (original, JPEG-75 re-encoded).
With 3,000 images per class this takes roughly 10-20 minutes on a T4. Results are printed and saved to Drive.

In [ ]:
import os
print("files on disk:", os.path.isdir("/content/diffface/SD21_T2I"))
print("variables alive:", all(v in globals() for v in ["FILES", "WORK", "real_paths", "fake_paths"]))

files on disk: True
variables alive: True


In [ ]:
import pandas as pd

PAIRINGS = {"A: DiffusionFace reals": real_paths}
if USE_FFHQ_REALS and ffhq_real_paths:
    PAIRINGS["B: FFHQ reals"] = ffhq_real_paths

VARIANTS = {"original": None, "jpeg75": 75}

rows = []
for model_name, ckpt_path in MODELS.items():
    if not os.path.exists(ckpt_path):
        print("Skipping (checkpoint not found):", model_name); continue
    model, cmap = load_resnet(ckpt_path)
    real_idx = cmap["real"]
    for pairing, reals in PAIRINGS.items():
        for fake_key in FAKE_KEYS:
            n = min(len(reals), len(fake_paths[fake_key]), n_eval)
            items = make_items(reals, fake_paths[fake_key], n)
            for vname, q in VARIANTS.items():
                p_real, y_real = score_items(model, real_idx, items, jpeg_q=q)
                m = metrics_from_scores(p_real, y_real)
                rows.append(dict(model=model_name, pairing=pairing, fake_set=fake_key, variant=vname,
                                 n_per_class=n, **m))
                print(f"{model_name:<26} {pairing:<24} {fake_key:<9} {vname:<9} "
                      f"acc {m['acc']:.3f}  AUC {m['auc']:.3f} [{m['auc_lo']:.3f}, {m['auc_hi']:.3f}]  "
                      f"fakes caught {m['fake_recall']:.3f}  reals kept {m['real_recall']:.3f}")
    del model
    torch.cuda.empty_cache()

results = pd.DataFrame(rows)
out_csv = f"{CHECKPOINT_DIR}/diffusionface_check_results.csv"
results.to_csv(out_csv, index=False)
print("\nSaved:", out_csv)
results.round(3)

Phase 2 (single-source)    A: DiffusionFace reals   SD15_T2I  original  acc 0.498  AUC 0.530 [0.515, 0.545]  fakes caught 0.013  reals kept 0.982
Phase 2 (single-source)    A: DiffusionFace reals   SD15_T2I  jpeg75    acc 0.494  AUC 0.521 [0.506, 0.536]  fakes caught 0.033  reals kept 0.955
Phase 2 (single-source)    A: DiffusionFace reals   SD21_T2I  original  acc 0.498  AUC 0.506 [0.490, 0.520]  fakes caught 0.014  reals kept 0.982
Phase 2 (single-source)    A: DiffusionFace reals   SD21_T2I  jpeg75    acc 0.496  AUC 0.520 [0.504, 0.534]  fakes caught 0.037  reals kept 0.955
Phase 2 (single-source)    B: FFHQ reals            SD15_T2I  original  acc 0.500  AUC 0.523 [0.508, 0.536]  fakes caught 0.013  reals kept 0.987
Phase 2 (single-source)    B: FFHQ reals            SD15_T2I  jpeg75    acc 0.496  AUC 0.505 [0.490, 0.519]  fakes caught 0.033  reals kept 0.960
Phase 2 (single-source)    B: FFHQ reals            SD21_T2I  original  acc 0.501  AUC 0.498 [0.484, 0.513]  fakes caught 0.

,model,pairing,fake_set,variant,n_per_class,acc,auc,auc_lo,auc_hi,fake_recall,real_recall,bal_acc
0,Phase 2 (single-source),A: DiffusionFace reals,SD15_T2I,original,3000,0.498,0.530,0.515,0.545,0.013,0.982,0.498
1,Phase 2 (single-source),A: DiffusionFace reals,SD15_T2I,jpeg75,3000,0.494,0.521,0.506,0.536,0.033,0.955,0.494
2,Phase 2 (single-source),A: DiffusionFace reals,SD21_T2I,original,3000,0.498,0.506,0.490,0.520,0.014,0.982,0.498
3,Phase 2 (single-source),A: DiffusionFace reals,SD21_T2I,jpeg75,3000,0.496,0.520,0.504,0.534,0.037,0.955,0.496
4,Phase 2 (single-source),B: FFHQ reals,SD15_T2I,original,3000,0.500,0.523,0.508,0.536,0.013,0.987,0.500
5,Phase 2 (single-source),B: FFHQ reals,SD15_T2I,jpeg75,3000,0.496,0.505,0.490,0.519,0.033,0.960,0.496
6,Phase 2 (single-source),B: FFHQ reals,SD21_T2I,original,3000,0.501,0.498,0.484,0.513,0.014,0.987,0.501
7,Phase 2 (single-source),B: FFHQ reals,SD21_T2I,jpeg75,3000,0.498,0.504,0.490,0.520,0.037,0.960,0.498
8,Phase 6 (mixed-source),A: DiffusionFace reals,SD15_T2I,original,3000,0.471,0.459,0.444,0.474,0.046,0.896,0.471
9,Phase 6 (mixed-source),A: DiffusionFace reals,SD15_T2I,jpeg75,3000,0.466,0.460,0.446,0.475,0.176,0.757,0.466


### Step 11 — Compact summary table (AUC with 95% bootstrap interval)

In [ ]:
summary = results.copy()
summary["AUC [95% CI]"] = summary.apply(lambda r: f"{r['auc']:.3f} [{r['auc_lo']:.3f}, {r['auc_hi']:.3f}]", axis=1)
summary = summary.pivot_table(index=["model", "pairing", "fake_set"], columns="variant",
                              values="AUC [95% CI]", aggfunc="first")
summary

variant                                                                jpeg75  \
model                   pairing                fake_set                         
Phase 2 (single-source) A: DiffusionFace reals SD15_T2I  0.521 [0.506, 0.536]   
                                               SD21_T2I  0.520 [0.504, 0.534]   
                        B: FFHQ reals          SD15_T2I  0.505 [0.490, 0.519]   
                                               SD21_T2I  0.504 [0.490, 0.520]   
Phase 6 (mixed-source)  A: DiffusionFace reals SD15_T2I  0.460 [0.446, 0.475]   
                                               SD21_T2I  0.444 [0.430, 0.459]   
                        B: FFHQ reals          SD15_T2I  0.496 [0.483, 0.510]   
                                               SD21_T2I  0.479 [0.465, 0.494]   

variant                                                              original  
model                   pairing                fake_set                        
Phase 2 (single-source) A: DiffusionFace reals SD15_T2I  0.530 [0.515, 0.545]  
                                               SD21_T2I  0.506 [0.490, 0.520]  
                        B: FFHQ reals          SD15_T2I  0.523 [0.508, 0.536]  
                                               SD21_T2I  0.498 [0.484, 0.513]  
Phase 6 (mixed-source)  A: DiffusionFace reals SD15_T2I  0.459 [0.444, 0.474]  
                                               SD21_T2I  0.466 [0.451, 0.482]  
                        B: FFHQ reals          SD15_T2I  0.498 [0.482, 0.511]  
                                               SD21_T2I  0.504 [0.490, 0.519]

## How to read the results (do not draw conclusions yet)

- **AUC around 0.5**: no usable signal; **below 0.5**: the model ranks fakes as more real than reals (as in Phase 7);
  **near 1.0**: the model detects this diffusion family.
- **Pairing A vs B**: if AUC is much higher with FFHQ reals (B) than with DiffusionFace reals (A), part of the failure is due to
  unfamiliar real photos, not only unfamiliar fakes (as in Phase 12).
- **original vs jpeg75**: a large change suggests the score depends on compression, not only on content.
- **Accuracy** is meaningful here because each evaluation has the same number of real and fake images.
- If Step 7 showed a file-facts AUC above 0.9, these results carry the same shortcut caveat as the old Phase 7.

**What to send back:** the Step 5 output, the Step 7 output, and the Step 10 printout (or the CSV).
Nothing in this notebook modifies Phase 7 or any checkpoint.